# Expressions in QProgram

QProgram has a **symbolic expression** system.  Anywhere a numeric
parameter is accepted — operation arguments, waveform parameters,
loop sources — you can pass a `Variable`, a `Constant`, or any
composition of them.  This notebook walks through the whole AST and
shows how to use each piece.

By the end you'll know:

- what kinds of nodes the AST has and how to construct each
- which operators are overloaded vs. exposed as named helpers
- how `evaluate()` works and how the `UNASSIGNED` sentinel
  propagates
- the two intentional gotchas (`==`/`!=` for variables, and
  `bool(expression)`)
- how expressions round-trip through the `.qp` file format

## 1. The expression hierarchy

Every node here is a subclass of `Expression`.

```
Expression                              ← abstract base
├── Leaves
│   ├── Variable                        identity-based, holds a value
│   └── Constant                        structural, holds a number
├── Arithmetic
│   ├── BinaryOp        + - * /
│   └── UnaryOp         unary - / +
├── Comparison & logical
│   ├── Comparison      == != < <= > >=     evaluates to bool
│   ├── LogicalBinaryOp and / or
│   └── LogicalNot      not
└── Math & conditional
    ├── MathFunc        sin cos tan exp log sqrt abs minimum maximum
    └── Where           ternary select
```

Some operators are reached via Python syntax (`+`, `<`, `&`, `abs`);
others are exposed as **module-level helpers** because Python's
syntax can't reach them or doing so would conflict with another
contract.

In [1]:
import qprogram as qp
from qprogram import (
    Variable, Constant, UNASSIGNED,
    # Comparison helpers (Python's == and != on Variable are identity-based,
    # so equality comparisons go through these named functions instead.)
    eq, ne,
    # Logical helpers — also reachable via `&`, `|`, `~`.
    and_, or_, not_,
    # Math functions.
    sin, cos, tan, exp, log, sqrt, minimum, maximum,
    # Ternary select.
    where,
)

## 2. Variables and Constants — the leaves

`Variable` is the symbolic placeholder; it holds a runtime value
(initially `UNASSIGNED`).  `Constant` is a concrete number.  Most
users never build `Constant` directly — it's auto-wrapped from
literals like `5` or `1e6` when arithmetic operators see them.

In [2]:
freq = Variable("freq", label="Drive frequency", units="Hz")
gain = Variable("gain")

print("freq.id     :", freq.id)
print("freq.label  :", freq.label)
print("freq.units  :", freq.units)
print("freq.value  :", freq.value, "  ← UNASSIGNED until set")
print()
# A Variable is its own variables-set; useful for static analysis.
print("freq.variables() :", freq.variables())

freq.id     : freq
freq.label  : Drive frequency
freq.units  : Hz
freq.value  : UNASSIGNED   ← UNASSIGNED until set

freq.variables() : {Variable('freq')}


### Identity-based equality, structural for everything else

Variables compare by identity (each `Variable("freq")` is distinct).
Every other node — `Constant`, `BinaryOp`, `Comparison`, … — uses
**structural** equality.  This matters because `expression.variables()`
returns a `set[Variable]`, and sets rely on `__hash__`/`__eq__`.

In [3]:
v1 = Variable("x")
v2 = Variable("x")
print("Same instance? ", v1 is v2)
print("v1 == v1       ", v1 == v1)
print("v1 == v2       ", v1 == v2, "  ← identity, not value")
print()
print("Constant(5) == Constant(5):", Constant(5) == Constant(5), "  ← structural")

Same instance?  False
v1 == v1        True
v1 == v2        False   ← identity, not value

Constant(5) == Constant(5): True   ← structural


## 3. Arithmetic — the familiar operators

`+ - * /` and unary `-/+` all build expression trees.  Literals on
either side are auto-promoted to `Constant`.

In [4]:
amp = Variable("amp")

expr = (amp + 0.1) * 2 - 0.5
print("expr     :", repr(expr))
print("expr type:", type(expr).__name__)
print("free vars:", expr.variables())

expr     : (((Variable('amp') + Constant(0.1)) * Constant(2)) - Constant(0.5))
expr type: BinaryOp
free vars: {Variable('amp')}


### `evaluate()` — read the current numeric value

Each `Variable` carries its own current value.  `evaluate()` walks
the tree, reads each variable, and returns the result — or
`UNASSIGNED` if any variable in the expression is currently unbound.

In [5]:
print("Before binding :", expr.evaluate())          # UNASSIGNED

amp.set_value(0.3)
print("amp = 0.3       :", expr.evaluate())          # (0.3 + 0.1) * 2 - 0.5 = 0.3

amp.reset()
print("After reset    :", expr.evaluate())          # UNASSIGNED again

Before binding : UNASSIGNED
amp = 0.3       : 0.30000000000000004
After reset    : UNASSIGNED


### `evaluate_or_raise()` — when you must have a number

Same as `evaluate()` but raises `UnassignedVariableError` instead of
returning the sentinel.  Use this when downstream code can't
meaningfully handle `UNASSIGNED` (e.g., computing a waveform
envelope to plot).

In [6]:
from qprogram import UnassignedVariableError

try:
    expr.evaluate_or_raise()
except UnassignedVariableError as e:
    print("Raised:", e)
    print("Free variables:", e.free_variables)

Raised: Cannot evaluate expression (((Variable('amp') + Constant(0.1)) * Constant(2)) - Constant(0.5)): unassigned variable(s) {Variable('amp')}
Free variables: {Variable('amp')}


## 4. Comparisons

QProgram supports `==`, `!=`, `<`, `<=`, `>`, `>=`.  Four of them
are Python operators (`<`, `<=`, `>`, `>=`).  **Equality and
inequality use named helpers** (`qp.eq`, `qp.ne`) — see the box
below for why.

In [7]:
# Operator form for the orderings:
print(freq < 5e9)
print(freq <= 5e9)
print(freq > 4e9)
print(freq >= 1e6)

(Variable('freq') < Constant(5000000000.0))
(Variable('freq') <= Constant(5000000000.0))
(Variable('freq') > Constant(4000000000.0))
(Variable('freq') >= Constant(1000000.0))


In [8]:
# Named-function form for equality:
print(eq(freq, 5e9))
print(ne(freq, 0))

(Variable('freq') == Constant(5000000000.0))
(Variable('freq') != Constant(0))


> **Why aren't `==` / `!=` overloaded?**
>
> `Variable.__eq__` is identity-based (every `Variable` instance is
> distinct, regardless of label).  Variables flow into `set[Variable]`
> as the result of `expression.variables()`, and Python's `set`
> implementation relies on `__eq__` returning `bool` for membership
> checks.  If `var == 5` built a `Comparison` instead of returning
> `False`, every `set.add(var)` would silently misbehave.
>
> Trade-off: `<`/`<=`/`>`/`>=` overload cleanly (they don't conflict
> with collection semantics); `==`/`!=` go through helper functions
> to keep both contracts working.

### Comparisons evaluate to `bool`

In [9]:
amp.set_value(0.3)

print("amp < 0.5   :", (amp < 0.5).evaluate())
print("eq(amp, 0.3):", eq(amp, 0.3).evaluate())
print("ne(amp, 0)  :", ne(amp, 0).evaluate())

amp < 0.5   : True
eq(amp, 0.3): True
ne(amp, 0)  : True


## 5. Logical operators — `& | ~`

Python's `and`, `or`, and `not` keywords short-circuit and cannot
be overloaded.  The NumPy / SymPy / Pandas convention is to repurpose
the bitwise operators `&`, `|`, `~` for logical composition of
symbolic expressions — QProgram follows the same convention.  For
readers who prefer a named form, `qp.and_`, `qp.or_`, `qp.not_` do
the same thing.

In [10]:
p = freq > 4e9
q = freq < 6e9

print("p & q :", (p & q))
print("p | q :", (p | q))
print("~p    :", (~p))
print()
print("and_(p, q):", and_(p, q))
print("or_(p, q) :", or_(p, q))
print("not_(p)   :", not_(p))

p & q : ((Variable('freq') > Constant(4000000000.0)) and (Variable('freq') < Constant(6000000000.0)))
p | q : ((Variable('freq') > Constant(4000000000.0)) or (Variable('freq') < Constant(6000000000.0)))
~p    : (not (Variable('freq') > Constant(4000000000.0)))

and_(p, q): ((Variable('freq') > Constant(4000000000.0)) and (Variable('freq') < Constant(6000000000.0)))
or_(p, q) : ((Variable('freq') > Constant(4000000000.0)) or (Variable('freq') < Constant(6000000000.0)))
not_(p)   : (not (Variable('freq') > Constant(4000000000.0)))


> **Precedence trap**
>
> `&` / `|` bind *tighter* than comparison operators in Python:
>
> ```python
> # This parses as: freq < (4e9 & gain) > 0.5
> # which is almost never what you want.
> expr = freq < 4e9 & gain > 0.5
>
> # Always parenthesise comparisons before combining them:
> expr = (freq < 4e9) & (gain > 0.5)
> ```
>
> NumPy/SymPy users will recognise this — the named form
> `and_(p, q)` sidesteps it entirely.

## 6. Math functions

All accept a single `Expression` (or numeric literal, auto-wrapped).
`abs()` works via Python's built-in — `MathFunc("abs", x)` is built
through `Expression.__abs__`.

In [11]:
print("sin(freq) :", sin(freq))
print("cos(freq) :", cos(freq))
print("exp(amp)  :", exp(amp))
print("sqrt(amp) :", sqrt(amp))
print("log(amp)  :", log(amp))
print("abs(amp)  :", abs(amp))

sin(freq) : sin(Variable('freq'))
cos(freq) : cos(Variable('freq'))
exp(amp)  : exp(Variable('amp'))
sqrt(amp) : sqrt(Variable('amp'))
log(amp)  : log(Variable('amp'))
abs(amp)  : abs(Variable('amp'))


### `minimum` / `maximum` — and why not the built-in `min`/`max`?

Python's `min(a, b)` uses `<` internally and returns one of the
operands based on the comparison result.  On `Expression` values,
`<` builds a `Comparison` node which is always *truthy*, so
`min(var_a, var_b)` silently returns `var_a` every time — a quiet
bug.

Use `qp.minimum(...)` and `qp.maximum(...)` instead.  Both are
variadic.

In [12]:
amp.set_value(0.3)

print("minimum(amp, 0.5)        :", minimum(amp, 0.5).evaluate())
print("minimum(amp, 0.5, 0.1)   :", minimum(amp, 0.5, 0.1).evaluate())
print("maximum(amp, 0.5)        :", maximum(amp, 0.5).evaluate())
print("maximum(amp, 0.5, 0.1)   :", maximum(amp, 0.5, 0.1).evaluate())

minimum(amp, 0.5)        : 0.3
minimum(amp, 0.5, 0.1)   : 0.1
maximum(amp, 0.5)        : 0.5
maximum(amp, 0.5, 0.1)   : 0.5


## 7. Conditional — `where(condition, then, else_)`

Three-arm select.  The condition is any `Expression` that evaluates
to a boolean (typically a `Comparison` or a logical composition).
Branches are `Expression`s or numeric literals (auto-wrapped to
`Constant`).

**Only the chosen branch is evaluated.**  The unchosen branch can
contain unbound variables without making the whole expression
`UNASSIGNED`.

In [13]:
# Build a recovery-amplitude expression: if amp is too large,
# halve it; otherwise pass through.
adjusted = where(amp > 0.5, amp / 2, amp)
print("expression:", adjusted)
print()

for v in [0.3, 0.7, 1.0]:
    amp.set_value(v)
    print(f"  amp={v} -> {adjusted.evaluate()}")

expression: where((Variable('amp') > Constant(0.5)), (Variable('amp') / Constant(2)), Variable('amp'))

  amp=0.3 -> 0.3
  amp=0.7 -> 0.35
  amp=1.0 -> 0.5


In [14]:
# The unchosen branch may reference unassigned variables.
unused = Variable("unused")
flag = Variable("flag"); flag.set_value(1)

# ``unused`` never gets evaluated when flag == 1.
print(where(eq(flag, 1), flag, unused).evaluate())

1


## 8. `UNASSIGNED` propagation

Any unbound variable poisons the whole tree — except the unchosen
branch of a `where`.  This is intentional: the evaluator never
short-circuits beyond what `where` strictly needs, so unbound
variables surface as `UNASSIGNED` reliably during debugging
instead of being hidden by short-circuit logic.

In [15]:
amp.reset()

print("(amp + 5).evaluate()        :", (amp + 5).evaluate())
print("(amp < 5).evaluate()        :", (amp < 5).evaluate())
print("sin(amp).evaluate()         :", sin(amp).evaluate())
print("(amp & p).evaluate()        :", (and_(amp > 0, amp < 1)).evaluate())
print("where(amp<1, 0, 1).evaluate :", where(amp < 1, 0, 1).evaluate())

(amp + 5).evaluate()        : UNASSIGNED
(amp < 5).evaluate()        : UNASSIGNED
sin(amp).evaluate()         : UNASSIGNED
(amp & p).evaluate()        : UNASSIGNED
where(amp<1, 0, 1).evaluate : UNASSIGNED


## 9. The `__bool__` guard — a deliberate `TypeError`

Calling `bool()` on an `Expression` raises `TypeError`.  This
catches the most common bug introduced when symbolic expressions
meet Python control flow:

```python
if freq < 5e9:           # NOPE — would always be True without the guard
    program.play(...)
```

`freq < 5e9` builds a `Comparison` node; without the guard, the
`if` would just test whether that object is truthy (which it is —
non-None objects are truthy by default).  The author almost
certainly meant to *build* a conditional, not to *test* one.

The fix is `where(...)` for in-expression conditionals, or — once
the upcoming `program.if_(predicate)` block lands — an explicit
control-flow construct.

In [16]:
try:
    if freq < 5e9:
        pass
except TypeError as e:
    print("Caught:", e)

Caught: Expression has no truth value — use .evaluate()/.evaluate_or_raise() to compute it, or qprogram.where(cond, then, else_) to build a conditional expression.


### Construction-time guard for bool conditions

The `Where` constructor and the logical helpers also reject a plain
`bool` where an `Expression` is expected.  This catches the
equally-common variant: someone writes `where(var == 1, ...)`
forgetting that `var == 1` returns identity-based `False`, not a
`Comparison`.  The error message points at the right helper.

In [17]:
try:
    where(amp == 0.3, 0, 1)
except TypeError as e:
    print("Caught:", e)

Caught: Where condition must be an Expression; got bool — if you wrote `var == literal` or `var != literal`, use qprogram.eq(var, literal) / qprogram.ne(...) instead, since Variable's `==` returns identity-based bool, not a Comparison.


## 10. Where expressions are used in programs

Anywhere `int | Expression` or `float | Expression` is accepted —
operation arguments, waveform parameters, loop sources — you can
pass any expression you've built above.  The runtime executor sets
variable values per loop iteration; expressions then evaluate
automatically.

In [18]:
prog = qp.QProgram(label="expressions-demo", description="What expressions look like in a program")
freq_v  = prog.variable("freq", label="Drive frequency", units="Hz")
gain_v  = prog.variable("gain")
flag_v  = prog.variable("flag")

with prog.average(shots=100):
    with prog.for_loop(freq_v, 4e9, 6e9, 1e6):
        # Frequency offset: variable arithmetic
        prog.set_frequency("drive_q0", freq_v + 1e6)
        # Variable phase via a math function
        prog.set_phase("drive_q0", sin(freq_v))
        # Clamp gain
        prog.set_gain("drive_q0", minimum(gain_v, 0.5))
        # Wait by a symbolic duration
        prog.wait("drive_q0", abs(freq_v - 5e9))
        # Conditional offset
        prog.set_offset("flux_q0", where(gain_v > 0.5, gain_v, 0.0))
        # Compound predicate (just to show off — set_offset takes a number,
        # so this would type-error in real use; the AST accepts it.)
        prog.set_offset("flux_q0", and_(eq(flag_v, 1), gain_v > 0))

print(qp.dumps(prog))

#!QProgram 1.0

metadata:
  label: "expressions-demo"
  description: "What expressions look like in a program"

body:
  var freq label="Drive frequency" units="Hz"
  var gain
  var flag

  average 100:
    for freq in range(4000000000.0, 6000000000.0, 1000000.0):
      set_frequency "drive_q0" (freq + 1000000.0)
      set_phase "drive_q0" sin(freq)
      set_gain "drive_q0" minimum(gain, 0.5)
      wait "drive_q0" abs((freq - 5000000000.0))
      set_offset "flux_q0" where((gain > 0.5), gain, 0.0)
      set_offset "flux_q0" ((flag == 1) and (gain > 0))



## 11. Round-trip through `.qp`

Every expression node serializes to a stable text form and parses
back to the same tree.  Two patterns:

- **Parenthesised**: arithmetic, comparison, and binary logical
  ops emit as `(<left> <op> <right>)`; unary `-`/`+` as `(-x)`;
  `not` as `(not x)`.
- **Function-call**: math functions and `where` emit as
  `name(arg, ...)` — the same shape waveform constructors use.

The parser dispatches function-call tokens by name: known math
names and `where` produce expression nodes, everything else goes
to the waveform registry.

In [19]:
text = qp.dumps(prog)
reloaded = qp.loads(text)

print("round-trip stable:", qp.dumps(reloaded) == text)

round-trip stable: True


## 12. A worked example — measurement-style conditional

QProgram doesn't yet expose runtime conditionals as a control-flow
block, but the **expression AST is already wide enough to describe
them**.  Today you can build a parametric recovery pulse whose
amplitude depends on a measurement-like classifier value; once
`program.if_(predicate)` lands, the same predicates will be
reusable verbatim.

In [20]:
# Pretend ``state`` is a per-shot classified outcome (0 or 1).
# Compute a recovery amplitude that's non-zero only when state == 1.
state = Variable("state")

recovery = where(eq(state, 1), 0.8, 0.0)
print("expression:", recovery)
print()

for s in (0, 1):
    state.set_value(s)
    print(f"  state={s} -> recovery amp = {recovery.evaluate()}")

expression: where((Variable('state') == Constant(1)), Constant(0.8), Constant(0.0))

  state=0 -> recovery amp = 0.0
  state=1 -> recovery amp = 0.8


## 13. Cheat sheet

| I want… | Use |
| --- | --- |
| Build `a + b`, `a - b`, `a * b`, `a / b` | `a + b`, `a - b`, … |
| Build `-a`, `+a` | `-a`, `+a` |
| Compare `a < b`, `a <= b`, `a > b`, `a >= b` | `a < b`, … |
| Compare `a == b`, `a != b` | `qp.eq(a, b)`, `qp.ne(a, b)` |
| Logical and | `a & b`  or  `qp.and_(a, b)` |
| Logical or | `a \| b`  or  `qp.or_(a, b)` |
| Logical not | `~a`  or  `qp.not_(a)` |
| Math functions | `qp.sin(x)`, `qp.cos(x)`, `qp.tan(x)`, `qp.exp(x)`, `qp.log(x)`, `qp.sqrt(x)`, `abs(x)` |
| Min / max | `qp.minimum(a, b, …)`, `qp.maximum(a, b, …)` |
| Ternary select | `qp.where(cond, then, else_)` |
| Evaluate | `expr.evaluate()`  (returns `UNASSIGNED` if unbound) |
| Evaluate strictly | `expr.evaluate_or_raise()` |
| Get free variables | `expr.variables()` |

### Three gotchas to remember

1. `var == x` returns an identity-based `bool`, not a `Comparison`.
   Use `qp.eq(var, x)`.
2. `min(a, b)` from the built-in silently mis-behaves on
   expressions.  Use `qp.minimum(a, b)`.
3. `if some_expression:` raises `TypeError`.  Use `qp.where(...)`
   for in-expression conditionals.